In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import time
import warnings
warnings.filterwarnings('ignore')

# Configuration
MAX_RUNTIME_HOURS = 11.5
MAX_RUNTIME_SECONDS = MAX_RUNTIME_HOURS * 3600
EPOCHS = 25
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Using device: {DEVICE}")
print(f"Maximum runtime: {MAX_RUNTIME_HOURS} hours")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Create results directory
Path("results").mkdir(exist_ok=True)

# ==================== DATASET ====================
def get_data_loaders():
    try:
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])
        
        trainset = torchvision.datasets.CIFAR10(
            root='./data', train=True, download=True, transform=transform
        )
        testset = torchvision.datasets.CIFAR10(
            root='./data', train=False, download=True, transform=transform
        )
        
        trainloader = DataLoader(
            trainset, batch_size=BATCH_SIZE, shuffle=True, 
            num_workers=4, pin_memory=True, persistent_workers=True
        )
        testloader = DataLoader(
            testset, batch_size=BATCH_SIZE, shuffle=False, 
            num_workers=4, pin_memory=True, persistent_workers=True
        )
        
        print(f"\n✓ Dataset loaded successfully")
        print(f"  Training samples: {len(trainset)}")
        print(f"  Test samples: {len(testset)}")
        print(f"  Batches per epoch: {len(trainloader)}")
        print(f"  Batch size: {BATCH_SIZE}")
        return trainloader, testloader
    except Exception as e:
        print(f"✗ Error loading dataset: {e}")
        raise

# ==================== VISION TRANSFORMER ====================
class PatchEmbedding(nn.Module):
    def __init__(self, img_size=32, patch_size=4, in_channels=3, embed_dim=192):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)
        
    def forward(self, x):
        x = self.proj(x)
        x = x.flatten(2).transpose(1, 2)
        return x

class MultiHeadAttention(nn.Module):
    def __init__(self, dim, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)
        
    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        return x

class MLP(nn.Module):
    def __init__(self, dim, hidden_dim):
        super().__init__()
        self.fc1 = nn.Linear(dim, hidden_dim)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_dim, dim)
        
    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.fc2(x)
        return x

class TransformerBlock(nn.Module):
    def __init__(self, dim, num_heads, mlp_ratio=4.0, use_norm=True, use_res=True):
        super().__init__()
        self.use_norm = use_norm
        self.use_res = use_res
        
        self.norm1 = nn.LayerNorm(dim) if use_norm else nn.Identity()
        self.attn = MultiHeadAttention(dim, num_heads)
        self.norm2 = nn.LayerNorm(dim) if use_norm else nn.Identity()
        self.mlp = MLP(dim, int(dim * mlp_ratio))
        
    def forward(self, x):
        if self.use_res:
            x = x + self.attn(self.norm1(x))
            x = x + self.mlp(self.norm2(x))
        else:
            x = self.attn(self.norm1(x))
            x = self.mlp(self.norm2(x))
        return x

class VisionTransformer(nn.Module):
    def __init__(self, img_size=32, patch_size=4, num_classes=10, embed_dim=192, 
                 depth=12, num_heads=3, mlp_ratio=4.0, use_norm=True, use_res=True):
        super().__init__()
        self.embed_dim = embed_dim
        self.patch_embed = PatchEmbedding(img_size, patch_size, 3, embed_dim)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, self.patch_embed.num_patches + 1, embed_dim))
        
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, mlp_ratio, use_norm, use_res)
            for _ in range(depth)
        ])
        
        self.norm = nn.LayerNorm(embed_dim) if use_norm else nn.Identity()
        self.head = nn.Linear(embed_dim, num_classes)
        
        self._init_weights()
        
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
                    
    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x)
        
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)
        x = x + self.pos_embed
        
        for block in self.blocks:
            x = block(x)
        
        x = self.norm(x)
        x = x[:, 0]
        x = self.head(x)
        return x

# ==================== METRICS LOGGER ====================
class MetricsLogger:
    def __init__(self, model, model_name):
        self.model = model
        self.model_name = model_name
        self.metrics = []
        self.hooks = []
        self.activations = {}
        
        # Register hooks for ALL MLP and Attention outputs
        for idx, block in enumerate(model.blocks):
            # Hook for attention output
            attn_hook = block.attn.register_forward_hook(self._make_hook(f'attn_{idx}'))
            self.hooks.append(attn_hook)
            
            # Hook for MLP output
            mlp_hook = block.mlp.register_forward_hook(self._make_hook(f'mlp_{idx}'))
            self.hooks.append(mlp_hook)
    
    def _make_hook(self, name):
        def hook(module, input, output):
            # Detach and move to CPU to free GPU memory
            self.activations[name] = output.detach().cpu()
        return hook
    
    def compute_stable_rank(self, x):
        """Compute stable rank: ||X||_F^2 / ||X||_2^2"""
        try:
            # Flatten batch and sequence dimensions
            if x.dim() == 3:
                B, N, D = x.shape
                x = x.reshape(B * N, D)
            elif x.dim() == 2:
                pass  # Already 2D
            else:
                return 1.0
            
            # Handle zero or near-zero matrices
            if x.abs().max() < 1e-8:
                return 1.0
            
            # Compute singular values (faster without computing U and V)
            s = torch.linalg.svdvals(x.float())
            
            # Filter out very small singular values
            s = s[s > 1e-8]
            
            if len(s) == 0:
                return 1.0
            
            # Stable rank formula
            frobenius_norm_sq = (s ** 2).sum()
            spectral_norm_sq = s[0] ** 2
            
            if spectral_norm_sq < 1e-8:
                return 1.0
            
            stable_rank = (frobenius_norm_sq / spectral_norm_sq).item()
            return max(1.0, min(stable_rank, float(len(s))))  # Cap at number of dimensions
        
        except Exception as e:
            print(f"Warning: Error computing stable rank: {e}")
            return 1.0
    
    def compute_active_neurons(self, x):
        """Compute percentage of active neurons (> 0)"""
        try:
            return (x > 0).float().mean().item()
        except Exception as e:
            print(f"Warning: Error computing active neurons: {e}")
            return 0.0
    
    def compute_gradient_norm(self):
        """Compute L2 norm of gradients for MLP layers"""
        try:
            grad_norms = []
            for block in self.model.blocks:
                if hasattr(block.mlp.fc1, 'weight') and block.mlp.fc1.weight.grad is not None:
                    norm = block.mlp.fc1.weight.grad.norm().item()
                    grad_norms.append(norm)
            return np.mean(grad_norms) if grad_norms else 0.0
        except Exception as e:
            print(f"Warning: Error computing gradient norm: {e}")
            return 0.0
    
    def compute_singular_value_spectrum(self, x, top_k=10):
        """Compute top-k singular values normalized by largest"""
        try:
            if x.dim() == 3:
                B, N, D = x.shape
                x = x.reshape(B * N, D)
            
            if x.abs().max() < 1e-8:
                return [1.0] + [0.0] * (top_k - 1)
            
            s = torch.linalg.svdvals(x.float())
            s = s[s > 1e-8]
            
            if len(s) == 0:
                return [1.0] + [0.0] * (top_k - 1)
            
            # Normalize by largest
            s_normalized = (s / s[0]).cpu().numpy()
            
            # Return top-k
            result = s_normalized[:top_k].tolist()
            # Pad with zeros if needed
            while len(result) < top_k:
                result.append(0.0)
            
            return result
        except Exception as e:
            print(f"Warning: Error computing SV spectrum: {e}")
            return [1.0] + [0.0] * (top_k - 1)
    
    def log_metrics(self, epoch, train_loss, val_loss, val_acc):
        """Compute and log all metrics"""
        try:
            metrics_dict = {
                'model': self.model_name,
                'epoch': epoch,
                'train_loss': train_loss,
                'val_loss': val_loss,
                'val_acc': val_acc,
                'grad_norm': self.compute_gradient_norm()
            }
            
            # Compute metrics for each layer
            for name, activation in self.activations.items():
                layer_type, layer_idx = name.split('_')
                
                # Stable rank
                rank = self.compute_stable_rank(activation)
                metrics_dict[f'rank_{layer_type}_{layer_idx}'] = rank
                
                # Active neurons (only for MLP)
                if layer_type == 'mlp':
                    active = self.compute_active_neurons(activation)
                    metrics_dict[f'active_{layer_type}_{layer_idx}'] = active
                
                # Singular value spectrum for first and last layer
                if layer_idx == '0' or layer_idx == '11':
                    sv_spectrum = self.compute_singular_value_spectrum(activation, top_k=10)
                    for k, sv_val in enumerate(sv_spectrum):
                        metrics_dict[f'sv_{layer_type}_{layer_idx}_top{k}'] = sv_val
            
            self.metrics.append(metrics_dict)
            self.activations.clear()  # Clear to free memory
            
            return metrics_dict
        except Exception as e:
            print(f"Warning: Error logging metrics: {e}")
            import traceback
            traceback.print_exc()
            return None
    
    def get_dataframe(self):
        return pd.DataFrame(self.metrics)
    
    def cleanup(self):
        for hook in self.hooks:
            hook.remove()

# ==================== TRAINING FUNCTIONS ====================
def train_epoch(model, trainloader, optimizer, criterion, device):
    """Train for one full epoch"""
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    num_batches = 0
    
    try:
        for batch_idx, (data, target) in enumerate(trainloader):
            data, target = data.to(device, non_blocking=True), target.to(device, non_blocking=True)
            
            optimizer.zero_grad(set_to_none=True)
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            _, predicted = output.max(1)
            total += target.size(0)
            correct += predicted.eq(target).sum().item()
            num_batches += 1
        
        avg_loss = total_loss / num_batches if num_batches > 0 else 0.0
        accuracy = 100. * correct / total if total > 0 else 0.0
        return avg_loss, accuracy
    except Exception as e:
        print(f"Error in training: {e}")
        import traceback
        traceback.print_exc()
        return 0.0, 0.0

@torch.no_grad()
def validate(model, testloader, criterion, device):
    """Validate the model"""
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    try:
        for data, target in testloader:
            data, target = data.to(device, non_blocking=True), target.to(device, non_blocking=True)
            output = model(data)
            loss = criterion(output, target)
            
            total_loss += loss.item()
            _, predicted = output.max(1)
            total += target.size(0)
            correct += predicted.eq(target).sum().item()
        
        avg_loss = total_loss / len(testloader)
        accuracy = 100. * correct / total
        return avg_loss, accuracy
    except Exception as e:
        print(f"Error in validation: {e}")
        import traceback
        traceback.print_exc()
        return 0.0, 0.0

@torch.no_grad()
def compute_metrics_on_batch(model, dataloader, device, num_batches=5):
    """Run forward pass on a few batches to collect activations for metrics"""
    model.eval()
    try:
        for batch_idx, (data, target) in enumerate(dataloader):
            if batch_idx >= num_batches:
                break
            data = data.to(device, non_blocking=True)
            _ = model(data)  # Forward pass triggers hooks
    except Exception as e:
        print(f"Error computing metrics: {e}")

# ==================== PLOTTING FUNCTIONS ====================
def save_plots(all_metrics):
    """Generate and save all plots"""
    try:
        print("\n📊 Generating plots...")
        
        # Plot A: Heatmap (Waterfall) - MLP Ranks
        fig, axes = plt.subplots(2, 3, figsize=(20, 12))
        axes = axes.flatten()
        
        for idx, (model_name, df) in enumerate(all_metrics.items()):
            if len(df) == 0 or idx >= 6:
                continue
            ax = axes[idx]
            
            # Extract MLP rank columns
            rank_cols = [col for col in df.columns if col.startswith('rank_mlp_')]
            rank_cols_sorted = sorted(rank_cols, key=lambda x: int(x.split('_')[-1]))
            
            if len(rank_cols_sorted) == 0:
                continue
            
            rank_data = df[rank_cols_sorted].values.T
            
            sns.heatmap(rank_data, ax=ax, cmap='viridis', cbar_kws={'label': 'Stable Rank'},
                       xticklabels=df['epoch'].values[::5], yticklabels=range(len(rank_cols_sorted)))
            ax.set_title(f'{model_name} - MLP Rank Evolution', fontsize=12, fontweight='bold')
            ax.set_xlabel('Epoch')
            ax.set_ylabel('Layer Index')
        
        plt.tight_layout()
        plt.savefig('results/plot_A_heatmap_mlp.png', dpi=150, bbox_inches='tight')
        plt.close()
        print("  ✓ Plot A (MLP Heatmap) saved")
        
        # Plot B: Depth Slice (Line Plot) - MLP Ranks at Final Epoch
        fig, ax = plt.subplots(figsize=(14, 7))
        
        for model_name, df in all_metrics.items():
            if len(df) == 0:
                continue
            last_epoch = df.iloc[-1]
            rank_cols = sorted([col for col in df.columns if col.startswith('rank_mlp_')], 
                             key=lambda x: int(x.split('_')[-1]))
            if len(rank_cols) == 0:
                continue
            ranks = [last_epoch[col] for col in rank_cols]
            layers = list(range(len(ranks)))
            ax.plot(layers, ranks, marker='o', label=model_name, linewidth=2.5, markersize=7)
        
        ax.set_xlabel('Layer Index', fontsize=13)
        ax.set_ylabel('Stable Rank', fontsize=13)
        ax.set_title('MLP Stable Rank by Layer (Final Epoch)', fontsize=15, fontweight='bold')
        ax.legend(fontsize=11)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig('results/plot_B_depth_slice.png', dpi=150, bbox_inches='tight')
        plt.close()
        print("  ✓ Plot B (Depth Slice) saved")
        
        # Plot C: Rank vs Sparsity (All Naked variants)
        naked_models = [name for name in all_metrics.keys() if 'Naked' in name]
        
        if len(naked_models) > 0:
            fig, axes = plt.subplots(1, len(naked_models), figsize=(8*len(naked_models), 6))
            if len(naked_models) == 1:
                axes = [axes]
            
            for idx, model_name in enumerate(naked_models):
                df = all_metrics[model_name]
                if len(df) == 0:
                    continue
                
                ax1 = axes[idx]
                
                # Average MLP rank
                rank_cols = [col for col in df.columns if col.startswith('rank_mlp_')]
                if len(rank_cols) > 0:
                    df['avg_rank'] = df[rank_cols].mean(axis=1)
                else:
                    continue
                
                # Average active neurons
                active_cols = [col for col in df.columns if col.startswith('active_mlp_')]
                if len(active_cols) > 0:
                    df['avg_active'] = df[active_cols].mean(axis=1)
                else:
                    df['avg_active'] = 0.5
                
                ax1.plot(df['epoch'], df['avg_rank'], 'b-', label='Avg Stable Rank', linewidth=2.5)
                ax1.set_xlabel('Epoch', fontsize=12)
                ax1.set_ylabel('Average Stable Rank', color='b', fontsize=12)
                ax1.tick_params(axis='y', labelcolor='b')
                
                ax2 = ax1.twinx()
                ax2.plot(df['epoch'], df['avg_active'] * 100, 'r-', label='Active Neurons %', linewidth=2.5)
                ax2.set_ylabel('Active Neurons (%)', color='r', fontsize=12)
                ax2.tick_params(axis='y', labelcolor='r')
                
                ax1.set_title(f'{model_name}: Rank vs Sparsity', fontsize=13, fontweight='bold')
                ax1.grid(True, alpha=0.3)
            
            fig.tight_layout()
            plt.savefig('results/plot_C_rank_vs_sparsity.png', dpi=150, bbox_inches='tight')
            plt.close()
            print("  ✓ Plot C (Rank vs Sparsity) saved")
        
        # Plot D: Accuracy Validation
        fig, ax = plt.subplots(figsize=(14, 7))
        
        for model_name, df in all_metrics.items():
            if len(df) > 0:
                ax.plot(df['epoch'], df['val_acc'], marker='o', label=model_name, linewidth=2.5, markersize=5)
        
        ax.set_xlabel('Epoch', fontsize=13)
        ax.set_ylabel('Validation Accuracy (%)', fontsize=13)
        ax.set_title('Validation Accuracy Over Time', fontsize=15, fontweight='bold')
        ax.legend(fontsize=11)
        ax.grid(True, alpha=0.3)
        ax.axhline(y=10, color='gray', linestyle='--', alpha=0.5, label='Chance Level')
        plt.tight_layout()
        plt.savefig('results/plot_D_accuracy.png', dpi=150, bbox_inches='tight')
        plt.close()
        print("  ✓ Plot D (Accuracy) saved")
        
        # Plot E: Singular Value Spectrum (First and Last Layer)
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        
        for model_name, df in all_metrics.items():
            if len(df) == 0:
                continue
            
            last_epoch = df.iloc[-1]
            
            # Layer 0 MLP
            sv_cols_0 = [col for col in df.columns if col.startswith('sv_mlp_0_top')]
            if len(sv_cols_0) > 0:
                sv_vals = [last_epoch[col] for col in sorted(sv_cols_0, key=lambda x: int(x.split('top')[-1]))]
                axes[0, 0].plot(range(len(sv_vals)), sv_vals, marker='o', label=model_name, linewidth=2)
            
            # Layer 11 MLP
            sv_cols_11 = [col for col in df.columns if col.startswith('sv_mlp_11_top')]
            if len(sv_cols_11) > 0:
                sv_vals = [last_epoch[col] for col in sorted(sv_cols_11, key=lambda x: int(x.split('top')[-1]))]
                axes[0, 1].plot(range(len(sv_vals)), sv_vals, marker='o', label=model_name, linewidth=2)
            
            # Layer 0 Attention
            sv_cols_0_attn = [col for col in df.columns if col.startswith('sv_attn_0_top')]
            if len(sv_cols_0_attn) > 0:
                sv_vals = [last_epoch[col] for col in sorted(sv_cols_0_attn, key=lambda x: int(x.split('top')[-1]))]
                axes[1, 0].plot(range(len(sv_vals)), sv_vals, marker='o', label=model_name, linewidth=2)
            
            # Layer 11 Attention
            sv_cols_11_attn = [col for col in df.columns if col.startswith('sv_attn_11_top')]
            if len(sv_cols_11_attn) > 0:
                sv_vals = [last_epoch[col] for col in sorted(sv_cols_11_attn, key=lambda x: int(x.split('top')[-1]))]
                axes[1, 1].plot(range(len(sv_vals)), sv_vals, marker='o', label=model_name, linewidth=2)
        
        axes[0, 0].set_title('Layer 0 MLP - Singular Value Spectrum', fontweight='bold')
        axes[0, 1].set_title('Layer 11 MLP - Singular Value Spectrum', fontweight='bold')
        axes[1, 0].set_title('Layer 0 Attention - Singular Value Spectrum', fontweight='bold')
        axes[1, 1].set_title('Layer 11 Attention - Singular Value Spectrum', fontweight='bold')
        
        for ax in axes.flatten():
            ax.set_xlabel('Singular Value Index')
            ax.set_ylabel('Normalized Value (σ_i / σ_1)')
            ax.legend()
            ax.grid(True, alpha=0.3)
            ax.set_ylim([-0.05, 1.05])
        
        plt.tight_layout()
        plt.savefig('results/plot_E_singular_values.png', dpi=150, bbox_inches='tight')
        plt.close()
        print("  ✓ Plot E (Singular Values) saved")
        
        print("✓ All plots generated successfully!")
        
    except Exception as e:
        print(f"✗ Error generating plots: {e}")
        import traceback
        traceback.print_exc()

# ==================== MAIN EXPERIMENT ====================
def run_experiment():
    global_start_time = time.time()
    
    print("="*80)
    print(" TRANSFORMER COLLAPSE PROTOCOL - EXPERIMENT 1")
    print(" Full-Fledged Training with Comprehensive Metrics")
    print("="*80)
    
    # Load data
    try:
        trainloader, testloader = get_data_loaders()
    except Exception as e:
        print(f"Failed to load data: {e}")
        return
    
    # Model configurations
    configs = {
        'ViT_Star': {'use_norm': True, 'use_res': True, 'embed_dim': 192},
        'ViT_NoNorm': {'use_norm': False, 'use_res': True, 'embed_dim': 192},
        'ViT_NoRes': {'use_norm': True, 'use_res': False, 'embed_dim': 192},
        'ViT_Naked': {'use_norm': False, 'use_res': False, 'embed_dim': 192},
        'ViT_Naked_2x': {'use_norm': False, 'use_res': False, 'embed_dim': 384},  # Double width
    }
    
    all_metrics = {}
    
    for model_idx, (model_name, config) in enumerate(configs.items()):
        # Check time budget
        elapsed = time.time() - global_start_time
        if elapsed > MAX_RUNTIME_SECONDS:
            print(f"\n⏰ Time limit reached! Stopping and saving results...")
            break
        
        remaining_time = (MAX_RUNTIME_SECONDS - elapsed) / 3600
        print(f"\n{'='*80}")
        print(f" MODEL {model_idx+1}/5: {model_name}")
        print(f" Config: LayerNorm={config['use_norm']}, Residual={config['use_res']}, Embed_Dim={config['embed_dim']}")
        print(f" Remaining time: {remaining_time:.2f} hours")
        print(f"{'='*80}")
        
        try:
            # Initialize model
            num_heads = config['embed_dim'] // 64  # Keep head dimension at 64
            model = VisionTransformer(
                img_size=32, patch_size=4, num_classes=10,
                embed_dim=config['embed_dim'], depth=12, num_heads=num_heads, mlp_ratio=4.0,
                use_norm=config['use_norm'], use_res=config['use_res']
            ).to(DEVICE)
            
            total_params = sum(p.numel() for p in model.parameters())
            print(f"  Total parameters: {total_params:,}")
            
            optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
            criterion = nn.CrossEntropyLoss()
            logger = MetricsLogger(model, model_name)
            
            model_start_time = time.time()
            
            # Training loop
            for epoch in range(1, EPOCHS + 1):
                epoch_start = time.time()
                
                # Check time budget
                elapsed = time.time() - global_start_time
                if elapsed > MAX_RUNTIME_SECONDS:
                    print(f"\n⏰ Time limit reached at epoch {epoch}!")
                    break
                
                # Train full epoch
                train_loss, train_acc = train_epoch(model, trainloader, optimizer, criterion, DEVICE)
                
                # Compute metrics on a subset of data (to collect activations)
                compute_metrics_on_batch(model, trainloader, DEVICE, num_batches=10)
                
                # Validate
                val_loss, val_acc = validate(model, testloader, criterion, DEVICE)
                
                # Log metrics
                metrics = logger.log_metrics(epoch, train_loss, val_loss, val_acc)
                
                epoch_time = time.time() - epoch_start
                print(f"Epoch {epoch:2d}/{EPOCHS} | "
                      f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
                      f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:5.2f}% | "
                      f"Time: {epoch_time:.1f}s")
                
                # Periodic save
                if epoch % 5 == 0:
                    df = logger.get_dataframe()
                    df.to_csv(f'results/{model_name}_metrics.csv', index=False)
                    print(f"  💾 Checkpoint saved at epoch {epoch}")
            
            # Final save
            df = logger.get_dataframe()
            df.to_csv(f'results/{model_name}_metrics.csv', index=False)
            all_metrics[model_name] = df
            
            model_time = time.time() - model_start_time
            print(f"\n✓ {model_name} completed in {model_time/60:.2f} minutes")
            
            # Cleanup
            logger.cleanup()
            del model, optimizer, logger
            torch.cuda.empty_cache()
            
        except Exception as e:
            print(f"✗ Error with {model_name}: {e}")
            import traceback
            traceback.print_exc()
            continue
    
    # Generate final plots
    if all_metrics:
        save_plots(all_metrics)
        
        # Save combined metrics
        try:
            combined_df = pd.concat([df for df in all_metrics.values()], ignore_index=True)
            combined_df.to_csv('results/all_metrics.csv', index=False)
            print(f"\n✓ Combined metrics saved to results/all_metrics.csv")
        except Exception as e:
            print(f"Warning: Could not save combined metrics: {e}")
    
    total_time = time.time() - global_start_time
    print(f"\n{'='*80}")
    print(f" EXPERIMENT COMPLETED")
    print(f" Total Runtime: {total_time/3600:.2f} hours ({total_time/60:.1f} minutes)")
    print(f" Models trained: {len(all_metrics)}")
    print(f" Results saved in: ./results/")
    print(f"{'='*80}")
    
    # Print summary statistics
    print("\n📊 FINAL SUMMARY:")
    print("-" * 80)
    for model_name, df in all_metrics.items():
        if len(df) > 0:
            final_acc = df.iloc[-1]['val_acc']
            final_loss = df.iloc[-1]['val_loss']
            
            # Calculate average rank for last epoch
            rank_cols = [col for col in df.columns if col.startswith('rank_mlp_')]
            if len(rank_cols) > 0:
                final_avg_rank = df.iloc[-1][rank_cols].mean()
            else:
                final_avg_rank = 0.0
            
            print(f"{model_name:20s} | Final Acc: {final_acc:5.2f}% | "
                  f"Final Loss: {final_loss:.4f} | Avg Rank: {final_avg_rank:.2f}")
    print("-" * 80)

# ==================== RUN ====================
if __name__ == "__main__":
    try:
        run_experiment()
    except KeyboardInterrupt:
        print("\n\n⚠️  Experiment interrupted by user. Saving available results...")
        print("Results have been saved to ./results/")
    except Exception as e:
        print(f"\n\n✗ Fatal error: {e}")
        import traceback
        traceback.print_exc()
        print("\nPartial results may be available in ./results/")

Using device: cuda
Maximum runtime: 11.5 hours
CUDA available: True
GPU: Tesla P100-PCIE-16GB
VRAM: 17.06 GB
 TRANSFORMER COLLAPSE PROTOCOL - EXPERIMENT 1
 Full-Fledged Training with Comprehensive Metrics


100%|██████████| 170M/170M [00:15<00:00, 10.9MB/s]



✓ Dataset loaded successfully
  Training samples: 50000
  Test samples: 10000
  Batches per epoch: 391
  Batch size: 128

 MODEL 1/5: ViT_Star
 Config: LayerNorm=True, Residual=True, Embed_Dim=192
 Remaining time: 11.49 hours
  Total parameters: 5,362,762
Epoch  1/25 | Train Loss: 1.8658 | Train Acc: 30.86% | Val Loss: 1.5617 | Val Acc: 42.06% | Time: 53.9s
Epoch  2/25 | Train Loss: 1.4905 | Train Acc: 45.17% | Val Loss: 1.4129 | Val Acc: 48.54% | Time: 52.4s
Epoch  3/25 | Train Loss: 1.3520 | Train Acc: 50.59% | Val Loss: 1.2770 | Val Acc: 53.16% | Time: 51.9s
Epoch  4/25 | Train Loss: 1.2748 | Train Acc: 53.69% | Val Loss: 1.3153 | Val Acc: 51.23% | Time: 51.9s
Epoch  5/25 | Train Loss: 1.2328 | Train Acc: 55.03% | Val Loss: 1.2781 | Val Acc: 54.07% | Time: 52.6s
  💾 Checkpoint saved at epoch 5
Epoch  6/25 | Train Loss: 1.1860 | Train Acc: 56.85% | Val Loss: 1.2301 | Val Acc: 55.74% | Time: 52.3s
Epoch  7/25 | Train Loss: 1.1607 | Train Acc: 57.92% | Val Loss: 1.1791 | Val Acc: 57.1